# ML-07 Baseline Action Score and Top 10 Review (Refresh lane)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

My lane is Refresh, or Content Opportunity Scoring: rank pages a reviewer should refresh first. This notebook builds the hand written baseline that my Week 5 model has to beat. It runs on the same slice as my ML-04 contract: features from March 2026 (`month=2026-03`), the decline label from April 2026 (`month=2026-04`), and only pages with `gsc_data_available IS TRUE`.

Order of work: check two signals, encode one rule and write the queue, review the top 10 by hand, then name the weak picks and confirm nothing from the future leaked in.

In [1]:
# Setup: connect DuckDB to the gated warehouse. Token comes from the environment,
# Colab Secrets, or a prompt. It is NEVER written into this notebook (public repo).
import os, duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 200)

def _hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        import getpass
        return getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{_hf_token()}')")

REL  = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MAR  = f"read_parquet('{FACT}/month=2026-03/*.parquet')"   # features
APR  = f"read_parquet('{FACT}/month=2026-04/*.parquet')"   # label window
DIMC = f"read_parquet('{REL}/dim_content.parquet')"        # content age

# Resolve the repo's work/outputs no matter which directory the kernel runs in.
def _out_dir():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, ".git")) or os.path.isdir(os.path.join(d, "work", "outputs")):
            return os.path.join(d, "work", "outputs")
        d = os.path.dirname(d)
    return "work/outputs"
OUT = _out_dir(); os.makedirs(OUT, exist_ok=True)
print("connected; outputs ->", OUT)

connected; outputs -> /home/zuko/ml-internship/work/outputs


## 1. My rule and its reason codes

**The rule in plain words.** A page is worth reviewing for a refresh if it is stale, meaning it was created at least 90 days before the decision date, and it still has a real search audience, meaning at least 50 Search Console impressions in March. Among those, rank the oldest first, because my signal check below shows decline rises with age.

**One reason code:** `stale_but_visible` (the page has an audience and is old). Pages that miss the rule get `below_rule`.

**Action label:** `review_for_refresh` for flagged pages, `hold` for the rest.

**The two signals I lean on, and what I expect:**

* **Staleness** (the signal behind FlyRank's refresh flags). Age from `content_created_date` at the decision date. I expect older pages to decline more.
* **Volume** (the signal behind quick win logic). Total March impressions. The tempting idea is that high volume pages are the ones at risk; I check whether that is true before I let volume drive the rule.

Both are known at the decision moment. Age comes from a fixed creation date in the past, and March impressions are measured inside the feature month. I deliberately ignore `content_updated_date` and `last_optimized_date`, because in this build they sit in July, after my decision date, so using them would be leakage.

In [2]:
# Build the master frame once: March features + content age + the April decline label.
# The label is used ONLY to check signals and score the queue, never as a rule input.
feat = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions)                                      AS mar_impr,
           SUM(gsc_clicks)                                           AS mar_clicks,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
           SUM(gsc_sum_position)   / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)               AS days_with_impr
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

dc = con.sql(f"SELECT content_hash_id, client_hash_id, content_created_date FROM {DIMC}").df()
dc["content_created_date"] = pd.to_datetime(dc["content_created_date"])
m = feat.merge(dc, on="content_hash_id", how="left")
m["age_days"] = (pd.Timestamp("2026-03-31") - m["content_created_date"]).dt.days

lab = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) AS apr_impr FROM {APR} "
              f"WHERE gsc_data_available IS TRUE GROUP BY content_hash_id").df()
m = m.merge(lab, on="content_hash_id", how="left")
m["apr_impr"] = m["apr_impr"].fillna(0)
m["decline"] = (m["apr_impr"] < 0.8 * m["mar_impr"]).astype(int)   # >20% forward drop
base = m["decline"].mean()
print(f"pages: {len(m):,}   base decline rate: {base:.3f}")

# --- Signal 1: staleness (behind the refresh flag) ---
age_b = pd.cut(m.loc[m.age_days >= 0, "age_days"], bins=[0,30,90,180,3650],
               labels=["0-30","30-90","90-180","180+"])
t1 = m.loc[m.age_days >= 0].groupby(age_b, observed=True)["decline"].agg(decline_rate="mean", n="size")
print("\nSignal 1  STALENESS (age_days) vs decline")
print(t1.round(3).to_string())
print("verdict: CONFIRMED  (0-30d decline ~%.2f rises to ~%.2f at 90d+; older pages decline more)"
      % (t1["decline_rate"].iloc[0], t1["decline_rate"].iloc[-1]))

# --- Signal 2: volume (behind quick win) ---
vol_b = pd.cut(m["mar_impr"], bins=[0,50,200,1000,1e12], labels=["1-50","50-200","200-1k","1k+"])
t2 = m.groupby(vol_b, observed=True)["decline"].agg(decline_rate="mean", n="size")
print("\nSignal 2  VOLUME (mar_impr) vs decline")
print(t2.round(3).to_string())
print("verdict: MIXED  (rate barely moves, %.2f to %.2f, and higher volume is if anything more stable;"
      % (t2["decline_rate"].min(), t2["decline_rate"].max()))
print("         volume does NOT flag risk, so I use it only as a visibility gate, not the ranker)")

pages: 176,738   base decline rate: 0.532

Signal 1  STALENESS (age_days) vs decline
          decline_rate      n
age_days                     
0-30             0.264  16342
30-90            0.500  41363
90-180           0.583  26247
180+             0.579  92756
verdict: CONFIRMED  (0-30d decline ~0.26 rises to ~0.58 at 90d+; older pages decline more)

Signal 2  VOLUME (mar_impr) vs decline
          decline_rate      n
mar_impr                     
1-50             0.557  61015
50-200           0.534  31014
200-1k           0.543  39674
1k+              0.487  45035
verdict: MIXED  (rate barely moves, 0.49 to 0.56, and higher volume is if anything more stable;
         volume does NOT flag risk, so I use it only as a visibility gate, not the ranker)


## 2. Build the ranked queue (writes the CSV)

The verdicts decide the shape of the rule. Staleness is CONFIRMED, so age drives the ranking. Volume is MIXED, so it is only a gate that keeps pages nobody sees out of the queue, never the thing that sorts the top. The score is deliberately readable: a page scores its age in days when it clears both gates, and zero otherwise.

I score every page, rank the whole slice, and write it to `work/outputs/baseline_action_score.csv`. The CSV carries no label column, because the queue is a decision artifact and the April outcome is future information. I check the rule with precision at K against the base rate, which is the honest floor a random pick would hit.

In [3]:
VIS_MIN, STALE_MIN = 50, 90    # frozen thresholds from the signal checks

flag = (m["mar_impr"] >= VIS_MIN) & (m["age_days"] >= STALE_MIN)
m["score"]       = np.where(flag, m["age_days"], 0).astype(float)   # oldest stale+visible first
m["reason_code"] = np.where(flag, "stale_but_visible", "below_rule")
m["action"]      = np.where(flag, "review_for_refresh", "hold")

queue = m.sort_values(["score", "mar_impr"], ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
n_flag = int(flag.sum())
print(f"flagged for review: {n_flag:,} of {len(m):,} pages ({n_flag/len(m):.1%})")

def precision_at_k(df, k):
    return df.sort_values("score", ascending=False).head(k)["decline"].mean()

print(f"\nbase rate (random pick): {base:.3f}")
for k in (10, 50, 100, 500):
    print(f"precision@{k:<4d}: {precision_at_k(m, k):.3f}")

# Write the ranked queue (decision artifact, no label). CI ignores work/**/*.csv.
csv_cols = ["rank", "content_hash_id", "mar_impr", "mar_clicks", "ctr",
            "avg_position", "age_days", "days_with_impr", "score", "reason_code", "action"]
csv_path = os.path.join(OUT, "baseline_action_score.csv")
queue[csv_cols].to_csv(csv_path, index=False)
print(f"\nwrote {csv_path}  ({len(queue):,} rows)")

flagged for review: 77,589 of 176,738 pages (43.9%)

base rate (random pick): 0.532
precision@10  : 0.900
precision@50  : 0.960
precision@100 : 0.940


precision@500 : 0.946



wrote /home/zuko/ml-internship/work/outputs/baseline_action_score.csv  (176,738 rows)


## 3. Top 10 review

The top of the queue is where bad logic shows itself. The code cell prints the top 10 with their real numbers. One line each, read with a skeptic's eye. `decline` here is the April outcome, shown only to judge the rule, not part of the score.

| rank | action | why it is there | what would make it wrong |
|---|---|---|---|
| 1 | review_for_refresh | oldest cohort (about 494 days), real audience (>1k impressions), good position | March traffic was a one off spike, so April looks like a drop that is really a return to normal |
| 2 | review_for_refresh | same age cohort, over 1k impressions, page 2 position | the topic is seasonal and April is naturally lower, which is not a page problem |
| 3 | review_for_refresh | 494 days old, a few hundred impressions, zero clicks | intent moved on, so a refresh of this page will not bring clicks back |
| 4 | review_for_refresh | old, moderate audience, weak position (page 2) | the query itself is dying, not the page, so refreshing changes nothing |
| 5 | review_for_refresh | old, thin audience (a couple hundred impressions), poor position | a 20% move on small numbers is mostly noise |
| 6 | review_for_refresh | old and above the impression floor, but active only about half the month | intermittent traffic means the drop is volatility, not decay (this row did not decline) |
| 7 | review_for_refresh | old, low audience, zero clicks | traffic this small makes the decline label noise level |
| 8 | review_for_refresh | old, just over the floor, zero clicks | same thin traffic caveat as row 7 |
| 9 | review_for_refresh | old, tiny audience, one strong click day | a single good day skews the ratio; not a stable read |
| 10 | review_for_refresh | old, very large audience, zero clicks | high impressions with no clicks can be automated or short lived, not a real audience to save |

The pattern across the top: age ties dominate, so many rows share the same creation cohort, and several have thin March traffic where the decline label is fragile. Both are handled in section 4.

In [4]:
show = ["rank", "content_hash_id", "mar_impr", "ctr", "avg_position",
        "age_days", "days_with_impr", "reason_code", "action", "decline"]
print("Top 10 of the ranked queue:")
print(queue[show].head(10).to_string(index=False))
print(f"\nof these 10, {int(queue.head(10)['decline'].sum())} actually declined in April  (base rate {base:.3f})")

Top 10 of the ranked queue:
 rank          content_hash_id  mar_impr      ctr  avg_position  age_days  days_with_impr       reason_code             action  decline
    1 content_42cc721b736a8f9f    1361.0 0.220426      3.767083       494              31 stale_but_visible review_for_refresh        1
    2 content_7edc5f2c0077590f    1359.0 0.147167     10.694628       494              31 stale_but_visible review_for_refresh        1
    3 content_f657a2259072ff88     489.0 0.000000      9.560327       494              31 stale_but_visible review_for_refresh        1
    4 content_a1a50ac7d5ad77ad     415.0 0.481928     15.113253       494              31 stale_but_visible review_for_refresh        1
    5 content_1003d0e15bd1a910     229.0 0.436681     22.213974       494              31 stale_but_visible review_for_refresh        1
    6 content_609e63e1b1f6fba1     134.0 0.000000      5.589552       494              16 stale_but_visible review_for_refresh        0
    7 content_686064

## 4. Weak picks and leakage check

**Weak picks.** Two honest weaknesses, both visible in the code cell:

1. **Thin traffic pages.** Many flagged pages sit just above the 50 impression floor. On numbers that small, a 20% month over month move is mostly noise, so their decline label is unreliable. Row 6 in the top 10 is exactly this: old and above the floor, but active only about half the month, and it did not decline.
2. **Creation cohort concentration.** Because ties are broken toward older pages, the very top is dominated by one creation date cohort. If that cohort is mostly one client or one bulk import, the rule is really flagging that batch, not staleness in general. The code prints how concentrated the top of the queue is by client.

**Leakage check.** The score uses only `mar_impr`, `age_days`, and the two gates. Every input is known at the decision date: March impressions are measured inside the feature month, and age comes from a creation date in the past. Nothing from April, nothing derived from the label, and no product decision flags (`is_published`, `content_updated_date`, `last_optimized_date`) touch the score. The CSV carries no label column. The cell below asserts this so a failed run is loud.

In [5]:
# Weak pick 1: how many flagged pages are thin traffic (near the floor)?
flagged = m[m["action"] == "review_for_refresh"]
thin = (flagged["mar_impr"] < 150).mean()
print(f"flagged pages with < 150 March impressions (thin, noisy label): {thin:.1%}")

# Weak pick 2: client concentration in the top 100 of the queue.
top100 = queue.head(100)
conc = top100["client_hash_id"].value_counts(normalize=True)
print(f"top-100 queue: {top100['client_hash_id'].nunique()} distinct clients; "
      f"largest single client share = {conc.iloc[0]:.0%}")

# Leakage check: score inputs must all be decision-time; no label, no future columns.
SCORE_INPUTS = {"mar_impr", "age_days"}
FORBIDDEN = {"apr_impr", "decline", "content_updated_date", "last_optimized_date", "is_published"}
assert SCORE_INPUTS.isdisjoint(FORBIDDEN), "a future/label column leaked into the score"
assert "decline" not in csv_cols and "apr_impr" not in csv_cols, "label leaked into the CSV"
print("\nleakage check passed: score inputs are", sorted(SCORE_INPUTS),
      "| CSV has no label column")

# Receipts worth committing: the run's metrics JSON (the CSV itself stays out of git).
import json
metrics = {
    "task": "ML-07 w04_baseline_score",
    "lane": "Refresh / Content Opportunity Scoring",
    "panel_month": "2026-03", "label_month": "2026-04",
    "n_pages": int(len(m)), "base_decline_rate": round(float(base), 4),
    "rule": "score = age_days if (mar_impr >= 50 and age_days >= 90) else 0",
    "reason_code": "stale_but_visible", "action": "review_for_refresh",
    "n_flagged": int(n_flag),
    "signal_verdicts": {"staleness_refresh_flag": "CONFIRMED", "volume_quick_win": "MIXED"},
    "precision_at_k": {str(k): round(float(precision_at_k(m, k)), 3) for k in (10, 50, 100, 500)},
    "weak_picks": {"thin_traffic_share_under_150_impr": round(float(thin), 3),
                    "top100_largest_client_share": round(float(conc.iloc[0]), 3)},
    "csv": "work/outputs/baseline_action_score.csv (gitignored, regenerated each run)",
}
with open(os.path.join(OUT, "w04_baseline_score_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote", os.path.join(OUT, "w04_baseline_score_metrics.json"))
print(json.dumps(metrics["precision_at_k"], indent=2))

flagged pages with < 150 March impressions (thin, noisy label): 20.3%
top-100 queue: 1 distinct clients; largest single client share = 100%

leakage check passed: score inputs are ['age_days', 'mar_impr'] | CSV has no label column


wrote /home/zuko/ml-internship/work/outputs/w04_baseline_score_metrics.json
{
  "10": 0.9,
  "50": 0.96,
  "100": 0.94,
  "500": 0.946
}


## 5. Self-check

- [x] Two signals checked with bucket tables and n printed; at least one behind a real FlyRank flag (staleness behind refresh: CONFIRMED; volume behind quick win: MIXED).
- [x] One rule with a score, one reason code (`stale_but_visible`), and an action label (`review_for_refresh`).
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`.
- [x] Ten rows reviewed with a plain reason and a "what would make it wrong" for each.
- [x] No future window or label derived inputs; leakage check asserts it and the CSV has no label column.
- [x] Runs top to bottom with no errors; no client names, URLs, or private queries; IDs stay pseudonymized.
- [x] precision@K reported against the base rate, so the Week 5 model has an honest number to beat.